# ⚙️ Centralized Lakehouse Utilities, Configuration & Shared Schemas
This utility notebook provides centralized dev/prod environment awareness, explicit StructType schema definitions, data quality validation rules, audit logging, and conformed transformation helpers imported across all pipelines via `%run ./utilities`.

### 📌 Step 1: Python Environment Resolution & Compatibility Bootstrap
* **Purpose:** Resolves project root on `sys.path` and initializes Databricks runtime compatibility objects (`spark`, `dbutils`, `display`) across cloud and local execution environments.
* **Logic & Transformations:** Detects current working directory, inserts repository root into `sys.path`, and invokes `init_notebook_context(globals())`.
* **Inputs & Dependencies:** Local filesystem hierarchy or Databricks notebook environment.
* **Outputs & Medallion State:** `spark`, `dbutils`, and `display` guaranteed to be present and functional in calling notebook scope.

In [1]:
# 1. Dynamic Repo Root Resolution for Python Module Imports
import sys, os

repo_root = None

# Strategy A: Databricks notebook context
try:
    nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    full_path = f"/Workspace{nb_path}" if not nb_path.startswith("/Workspace") else nb_path
    parts = full_path.split("/")
    if "Atlikon_DE" in parts:
        idx = parts.index("Atlikon_DE")
        repo_root = "/".join(parts[:idx+1])
except Exception:
    pass

# Strategy B: Local file or CWD inspection
if not repo_root or not os.path.exists(os.path.join(repo_root, "src")):
    try:
        current_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        current_dir = os.getcwd()
    candidate = current_dir
    while candidate and candidate != "/":
        if os.path.exists(os.path.join(candidate, "src")):
            repo_root = candidate
            break
        candidate = os.path.dirname(candidate)

# Strategy C: Databricks Workspace fallback paths
if not repo_root or not os.path.exists(os.path.join(repo_root, "src")):
    for candidate in [
        "/Workspace/Users/veeranithin9@gmail.com/Atlikon_DE",
        "/Workspace/Repos/veeranithin9@gmail.com/Atlikon_DE",
    ]:
        if os.path.exists(os.path.join(candidate, "src")):
            repo_root = candidate
            break

if repo_root and repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Initialize Databricks local compatibility shim
try:
    from src.compat import init_notebook_context
    spark, dbutils, display = init_notebook_context(globals())
except (ImportError, Exception):
    pass


### 📌 Step 2: Environment-Aware Lakehouse Configuration (Dev vs Prod)
* **Purpose:** Parameterizes the target catalog, schema names, and AWS S3 storage buckets dynamically based on the active environment (`prod` vs `dev`).
* **Logic & Transformations:**
  1. Evaluates widget parameter `env` or `ENVIRONMENT` environment variable.
  2. Maps `dev` -> catalog `fmcg_dev`, S3 bucket `spartsbar-2355-dev`.
  3. Maps `prod` -> catalog `fmcg`, S3 bucket `spartsbar-2355`.
  4. Defines helper function `base_path_for(data_source)` returning target S3 landing globs.
* **Inputs & Dependencies:** `dbutils.widgets` or OS environment variables.
* **Outputs & Medallion State:** Configuration variables `catalog`, `bronze_schema`, `silver_schema`, `gold_schema`, `s3_bucket`, and helper `base_path_for`.

In [2]:
# 2. Environment-Aware Configuration (dev vs prod)
try:
    env = dbutils.widgets.get("env").lower()
except Exception:
    env = os.getenv("ENVIRONMENT", "prod").lower()

CONFIG = {
    "dev": {"catalog": "fmcg_dev", "bucket": "spartsbar-2355-dev"},
    "prod": {"catalog": "fmcg", "bucket": "spartsbar-2355"},
}
active_config = CONFIG.get(env, CONFIG["prod"])
catalog = active_config["catalog"]
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"
s3_bucket = active_config["bucket"]

def base_path_for(data_source: str) -> str:
    return f"s3://{s3_bucket}/{data_source}/*.csv"


### 📌 Step 3: Expose Conformed Schemas, Transformation Functions & Quality Rules
* **Purpose:** Exports modular, test-verified business logic components from `src/` into the global notebook namespace for seamless pipeline invocation.
* **Logic & Transformations:** Imports:
  * Explicit Schemas: `orders_schema`, `products_schema`, `pricing_schema`, `customers_schema`.
  * Transformations: `clean_sentinel_id`, `strip_weekday_prefix`, `parse_multi_format_date`, `clean_price_column`, `extract_product_variant`, `generate_product_hash`, `clean_cities`, `clean_orders_silver`, `aggregate_orders_to_monthly`.
  * Data Quality: `run_quality_checks`, expectation dictionaries for all 4 pipelines.
  * Audit Logging: `add_audit_columns`, `log_pipeline_event`, `generate_run_id`.
* **Inputs & Dependencies:** Python modules under `src/`.
* **Outputs & Medallion State:** Complete toolkit of enterprise lakehouse functions available to any downstream notebook.

In [3]:
# 3. Expose Schemas, Transformations, Data Quality & Auditing
from src.schemas import (
    orders_schema, products_schema, pricing_schema, customers_schema
)
from src.transformations import (
    clean_sentinel_id, strip_weekday_prefix, parse_multi_format_date,
    clean_price_column, extract_product_variant, generate_product_hash,
    clean_cities, clean_orders_silver, aggregate_orders_to_monthly
)
from src.data_quality import (
    run_quality_checks, ORDERS_QUALITY_CHECKS, PRICING_QUALITY_CHECKS,
    PRODUCTS_QUALITY_CHECKS, CUSTOMERS_QUALITY_CHECKS
)
from src.audit import add_audit_columns, log_pipeline_event, generate_run_id
